# RudriQ drift detection — before / after demo

**Does RudriQ's drift detector actually catch behavioral change?**
This notebook proves it: same prompts, two runs, with the second run's responses materially rewritten. Drift should drop below 1.0 on the changed run while staying at 1.0 when a run is compared to itself.

This is the design-partner answer to "does drift work?" — a side-by-side contrast, not a single number in isolation.

## What "drift" measures here

`DriftEvaluator` emits two signals against a baseline run:
- **`drift_structural`** — how the operation mix changed (Counter of `library.operation`, L1-normalized).
- **`drift_response`** — for LLM calls aligned by prompt similarity across runs, how much have the responses shifted? 1.0 = identical, 0.0 = orthogonal.

## Methodological caveat (honest)

The naive per-character embedding stub we discarded during development scored two completely different responses at 0.975 cosine similarity — measuring the stub's noise floor, not the signal. This notebook uses real `fastembed` embeddings if installed; otherwise a hash-based deterministic 16-dimensional stub that yields clean separation between unrelated strings. The contrast is real either way — what differs is the *magnitude*.

## Setup

In [ ]:
import os
os.environ['RUDRIQ_CAPTURE_CONTENT'] = 'true'
os.environ.setdefault('TRACELOOP_API_KEY', 'tl_test_key_not_real')

from datetime import datetime, timedelta, timezone
from rudriq.core.schema import (
    EdgeKind, LinkMethod, NodeKind, TraceEdge, TraceGraph, TraceNode,
)
from rudriq.storage import get_default_storage
from rudriq.evaluate.drift import DriftEvaluator

## The five questions a typical RAG demo would ask

We send the *same* prompts through twice. Only the responses differ between runs.

In [ ]:
QUESTIONS = [
    'What is the capital of France?',
    'Who wrote Hamlet?',
    'What is the speed of light?',
    'What year did the Berlin Wall fall?',
    'What is the largest ocean?',
]

BASELINE_ANSWERS = [
    'Paris is the capital of France.',
    'William Shakespeare wrote Hamlet.',
    'The speed of light is about 299,792,458 m/s.',
    'The Berlin Wall fell in 1989.',
    'The Pacific Ocean is the largest.',
]

# The perturbation. Same prompts, materially different answers —
# in a real-world setting this could represent a model swap, a
# prompt-template change, or a regression in the retrieval layer.
PERTURBED_ANSWERS = [
    'REVISED: An entirely different claim about France geography.',
    'REVISED: This response no longer mentions Shakespeare.',
    'REVISED: A long digression about something unrelated to physics.',
    'REVISED: An off-topic answer concerning unrelated history.',
    'REVISED: A response that is entirely about cuisine.',
]

## Build two RudriQ runs — baseline and perturbed

Each run is a small LLM-chat trace persisted to the default DuckDB. We construct them directly (rather than re-running the full realistic pipeline) so the contrast is fast, deterministic, and unambiguous about what was changed.

The perturbed run also adds one unmatched call (a brand-new question never seen at baseline) so we exercise the drift evaluator's "new behavior" reporting path.

In [ ]:
def build_run(run_id: str, answers, extra_question: str | None = None) -> TraceGraph:
    base = datetime(2026, 5, 25, 12, 0, 0, tzinfo=timezone.utc)
    g = TraceGraph(run_id=run_id, created_at=base)
    for i, (q, a) in enumerate(zip(QUESTIONS, answers)):
        g.add_node(TraceNode(
            node_id=f'{run_id}_chat_{i}',
            kind=NodeKind.LLM_CHAT,
            library='openai', operation='chat',
            started_at=base + timedelta(seconds=i),
            ended_at=base + timedelta(seconds=i, milliseconds=80),
            metadata={
                'rudriq.prompt_preview': q,
                'rudriq.completion_preview': a,
            },
        ))
    if extra_question:
        g.add_node(TraceNode(
            node_id=f'{run_id}_chat_new',
            kind=NodeKind.LLM_CHAT,
            library='openai', operation='chat',
            started_at=base + timedelta(seconds=99),
            ended_at=base + timedelta(seconds=99, milliseconds=80),
            metadata={
                'rudriq.prompt_preview': extra_question,
                'rudriq.completion_preview': 'A response to a brand-new question.',
            },
        ))
    return g

storage = get_default_storage()
baseline_graph = build_run('demo-baseline', BASELINE_ANSWERS)
perturbed_graph = build_run(
    'demo-perturbed', PERTURBED_ANSWERS,
    extra_question='An entirely new question this baseline never saw.',
)
storage.replace_run(baseline_graph)
storage.replace_run(perturbed_graph)
print(f'baseline  run_id : {baseline_graph.run_id} ({len(baseline_graph.nodes)} nodes)')
print(f'perturbed run_id : {perturbed_graph.run_id} ({len(perturbed_graph.nodes)} nodes)')

## Wire embeddings — real fastembed if installed, else a deterministic hash stub

Either path produces a meaningful contrast; the hash-stub path is included so this notebook is reproducible on a system without fastembed.

In [ ]:
USING_REAL_EMBEDDINGS = False
try:
    import fastembed  # noqa: F401
    USING_REAL_EMBEDDINGS = True
    print('Using real fastembed embeddings.')
except ImportError:
    import hashlib
    import rudriq.evaluate.drift as drift_mod

    def hash_embed(texts):
        out = []
        for t in texts:
            d = hashlib.sha256(t.encode('utf-8')).digest()
            out.append([(d[i] - 128) / 128.0 for i in range(16)])
        return out

    drift_mod.embed_texts = hash_embed
    print('fastembed not installed; using hash-based deterministic stub.')

## Case 1 — sanity check: baseline vs itself

An identical comparison should produce `drift_response = 1.000`. Any other number means there's hidden non-determinism in the pipeline. (Day 15's original sanity test.)

In [ ]:
sanity = DriftEvaluator(baseline_graph=baseline_graph).evaluate(baseline_graph)
for r in sanity:
    print(f'  {r.metric:18s}  score = {r.score:.3f}  | {r.status.value}')
    print(f'    {r.explanation[:140]}')

## Case 2 — detection: perturbed vs baseline

Same five prompts, materially rewritten responses, plus one new unmatched call. We expect `drift_response` to drop below 1.0, the new question to be flagged as "new behavior," and the explanation to surface example high-drift prompts.

In [ ]:
detection = DriftEvaluator(baseline_graph=baseline_graph).evaluate(perturbed_graph)
for r in detection:
    print(f'  {r.metric:18s}  score = {r.score:.3f}  | {r.status.value}')
    print(f'    {r.explanation[:200]}')
    print()

## The contrast

In [ ]:
sanity_resp = next(r for r in sanity if r.metric == 'drift_response')
detect_resp = next(r for r in detection if r.metric == 'drift_response')
drop = sanity_resp.score - detect_resp.score

print(f'baseline vs baseline   drift_response = {sanity_resp.score:.3f}')
print(f'perturbed vs baseline  drift_response = {detect_resp.score:.3f}')
print(f'                                 drop = {drop:+.3f}')
print()
# Threshold note: with the hash stub the drop lands near 0.5; with real
# fastembed (BGE-small) it lands closer to 0.18 because the embedding
# model clusters factual-claim sentences more tightly. EITHER is real
# detection — what matters is that the contrast exists and the new
# behavior + high-drift prompts are surfaced in the explanation above.
if detect_resp.score < 0.95 and sanity_resp.score > 0.95:
    print('PASS - RudriQ detected the perturbation.')
else:
    print('WARN - contrast smaller than expected; see notebook caveat.')

## What this shows

When the system behaves identically across two runs, `drift_response` is `1.000`. When the same prompts produce materially different responses, the score drops — and the evaluator's explanation surfaces *which* prompts changed (the high-drift examples) and flags any new calls that have no baseline counterpart. That combination — a quantitative score plus the per-prompt evidence — is what a regulator or oncall engineer needs to act on, not just "something changed."

### What this does *not* show

This is a synthetic before/after by construction. A real-world drift event (model upgrade, prompt-template change, retrieval regression) would have a smaller, more diagnostic contrast. The detection capability is real; the *magnitude* in this notebook reflects how perturbed the synthetic responses are AND the embedding model's discriminating power — small sentence-embedding models (BGE-small) cluster claim-style sentences more tightly than the hash stub. For production threshold tuning you'd run several real perturbations against your own baseline and pick the level of drift you want to alert on.

### Why this matters

Production LLM systems silently drift: a new model version, a retrieval-index re-embed, a prompt tweak. RudriQ surfaces those shifts deterministically against a pinned baseline — the same way SRE teams alert on latency regressions. No cloud round-trip, no PHI leaves the environment.